In [2]:
import random

## Story of Langchain
AI engineers manually connecting components

In [3]:
class NakliLLM:
    
    def __init__(self):
        print('LLM created')
    
    def predict(self, prompt):
        response_list = [
            'Delhi is capital of india',
            'AI stands for something',
            'Cricket is a sport'
        ]
        
        return { 'response' : random.choice(response_list)}

In [7]:
class NakliPromptTemplate:
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables
    
    def format(self, input_dict):
        return self.template.format(**input_dict)

In [ ]:
llm = NakliLLM()
prompt = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length','topic']
)
final_prompt = prompt.format({'length':'short', 'topic':'india'})


LLM created


In [ ]:
llm.predict(final_prompt)

{'response': 'Cricket is a sport'}

### Now Langchain introduces Chains

In [11]:
class NakliLLMChain:
    def __init__(self, llm, prompt):
        self.llm = llm
        self.prompt = prompt
    
    def run(self, input_dict):
        prompt = self.prompt.format(input_dict)
        result = self.llm.predict(prompt)
        return result['response']

In [13]:
template = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length','topic']
)
llm = NakliLLM()

chain = NakliLLMChain(llm, template)

chain.run({
    'length':'short',
    'topic':'cricket'
})

LLM created


'Delhi is capital of india'

### Standardize Components using Runnable using Abstractions

In [16]:
from abc import ABC, abstractmethod

In [18]:
class Runnable(ABC):
    @abstractmethod
    def invoke(input_data):
        pass

In [69]:
class NakliLLM(Runnable):
    
    def __init__(self):
        print('LLM created')
    
    def invoke(self, prompt):
        response_list = [
            'Delhi is capital of india',
            'AI stands for something',
            'Cricket is a sport'
        ]
        
        return { 'response' : random.choice(response_list)}

    
    def predict(self, prompt):
        print("predict is going to be @deprecated")
        response_list = [
            'Delhi is capital of india',
            'AI stands for something',
            'Cricket is a sport'
        ]
        
        return { 'response' : random.choice(response_list)}

class NakliPromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables
    
    def invoke(self, input_dict):
        print(input_dict,type(input_dict))
        return self.template.format(**input_dict)
        
    def format(self, input_dict):
        print("format is going to be @deprecated")
        return self.template.format(**input_dict)

In [70]:
class RunnableConnector(Runnable):
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list
    
    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data

In [71]:
class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass
    def invoke(self, input_data):
        return input_data['response']

In [72]:
llm = NakliLLM()
template = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length','topic']
)
parser = NakliStrOutputParser()

LLM created


In [73]:
chain = RunnableConnector([template,llm,parser])

In [74]:
chain.invoke({
    'length':'short',
    'topic':'cricket'
})

{'length': 'short', 'topic': 'cricket'} <class 'dict'>


'Delhi is capital of india'

### Joining 2 Runnables

In [ ]:
template1 = NakliPromptTemplate(
    template='Write a joke about {topic}',
    input_variables='topic'
)
template2 = NakliPromptTemplate(
    template='Write a summary about {response}',
    input_variables='response'
)
llm = NakliLLM()
parser = NakliStrOutputParser()

LLM created


In [76]:
chain1 = RunnableConnector([template1,llm])
chain2 = RunnableConnector([template2,llm,parser])

In [77]:
chain1.invoke({
    'topic':'cricket'
})
chain2.invoke({
    'joke':'AI stands for something'
})

{'topic': 'cricket'} <class 'dict'>
{'joke': 'AI stands for something'} <class 'dict'>


'Cricket is a sport'

In [80]:
chain3 = RunnableConnector([chain1,chain2])

In [81]:
chain3.invoke({
    'topic':'cricket'
})

{'topic': 'cricket'} <class 'dict'>
{'response': 'Cricket is a sport'} <class 'dict'>


KeyError: 'joke'